In [3]:
import pandas as pd
import joblib
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from utils import fit_position_pipeline, select_features_by_correlation

In [4]:
# features = joblib.load('../predictors/hist/feats_fwd')

features = [
    "was_home",
    "difficulty",
    "opponent_difficulty",
    "win_prob",
    "elo_diff",
    "yellow_cards_rolling_5",
    "starts_rolling_3",
    "points_3_ewm",
    "xP_3_ewm",
    "minutes_rolling_3",
    "goals_scored_3_ewm",
    "assists_3_ewm",
    "ict_index_3_ewm",
    "expected_goals_3_ewm",
    "expected_assists_3_ewm",
    "expected_goal_involvements_3_ewm",
    "interceptions_3_ewm",
    "blocks_3_ewm",
    "clearances_3_ewm",
    "tackles_3_ewm",
    "selected_by_percent",
    "transfers_in",
]


player_data = pd.read_csv("../../inputs/rolled_data_24-25_xp_fixed.csv")
player_data = player_data[player_data["position"] == "Forward"]

league_models = {}

for gw in range(6, 39):
    print(f"\n{'=' * 60}\n Model for round {gw} \n{'=' * 60}")
    data = player_data[(player_data["round"] > 3) & (player_data["round"] < gw)].fillna(
        0
    )
    feats = features  # features[gw]
    # feats.remove('xP')
    selected_feats = select_features_by_correlation(
        data,
        feats,
        target_col="xP",
        redundancy_threshold=0.45,
        min_corr=0.15,
        verbose=True,
    )

    league_model = fit_position_pipeline(
        data, selected_feats, gw, position_name="FWD", random_seed=42
    )

    print(f"Features for FWD kept in gw {gw} -> {league_model['feats']}")

    league_models[gw] = league_model



 Model for round 19 
  dropping 'ict_index_3_ewm' (|corr| with xP=0.866) -- |corr|=0.937 with already-selected 'points_3_ewm'
  dropping 'xP_3_ewm' (|corr| with xP=0.865) -- |corr|=0.899 with already-selected 'points_3_ewm'
  dropping 'minutes_rolling_3' (|corr| with xP=0.802) -- |corr|=0.838 with already-selected 'points_3_ewm'
  dropping 'goals_scored_3_ewm' (|corr| with xP=0.785) -- |corr|=0.892 with already-selected 'points_3_ewm'
  dropping 'starts_rolling_3' (|corr| with xP=0.775) -- |corr|=0.809 with already-selected 'points_3_ewm'
  dropping 'expected_goal_involvements_3_ewm' (|corr| with xP=0.768) -- |corr|=0.844 with already-selected 'points_3_ewm'
  dropping 'expected_goals_3_ewm' (|corr| with xP=0.733) -- |corr|=0.808 with already-selected 'points_3_ewm'
  dropping 'transfers_in' (|corr| with xP=0.617) -- |corr|=0.686 with already-selected 'points_3_ewm'
  dropping 'selected_by_percent' (|corr| with xP=0.597) -- |corr|=0.614 with already-selected 'points_3_ewm'
  dropping 

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  divergences: 0
Kept 3 / 5 features: ['elo_diff', 'points_3_ewm', 'blocks_3_ewm']
Refitting on pruned feature set...


... (191 outputs left out)

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  divergences: 0
Features for FWD kept in gw 38 -> ['opponent_difficulty', 'points_3_ewm', 'blocks_3_ewm']


In [5]:
joblib.dump(league_models, "./league_models_fwd")

['./league_models_fwd']